In [1]:
import os
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "3"


import torch
# torch.cuda.init()
print(os.environ.get("CUDA_VISIBLE_DEVICES"))
!source /home/jupyter/Mrigi/env.sh
hf_token = os.environ.get("HF_TOKEN")

torch.cuda.set_device(0)
import json

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.vectorstores import FAISS
from langchain.prompts import ChatPromptTemplate
from langchain.llms import HuggingFacePipeline
from langchain.chains import LLMChain
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline



3


In [2]:
from pydantic import BaseModel, Field
from typing import Optional, Union
from datetime import datetime
from tqdm import tqdm

In [3]:
!nvidia-smi

Mon Feb  2 18:13:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.119.02             Driver Version: 580.119.02     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               Off |   00000000:31:00.0 Off |                  Off |
| 46%   75C    P2            180W /  230W |    2513MiB /  24564MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:

!kill -9 3649195

/bin/bash: line 0: kill: (3649195) - No such process


In [5]:
# ---------- Step 1: Load and Process the JSON Records ----------
# # Load JSON records from file
# with open('filtered_records_subset_100.json', 'r') as file:
#     filtered_records = json.load(file)

In [6]:
# Process each record: extract paragraphs with "Experimental" supersection,
# concatenate them, and append DOI info.
# documents = []
# metadata = []

In [7]:
# Process records into documents
# documents = []
# metadata = []

# for record in filtered_records:
#     doi = record.get("doi", "Unknown DOI")
    
#     # Add abstract as a separate document
#     abstract_text = record.get("abstract", "").strip()
#     if abstract_text:
#         combined_text = f"{abstract_text}\n\nThis information is from DOI: {doi}"
#         documents.append(combined_text)
#         metadata.append({"doi": doi, "source": "abstract"})
    
#     # Add each paragraph as a separate document
#     for para in record.get("paragraphs", []):
#         paragraph_text = para.get("text", "").strip()
#         if paragraph_text:
#             combined_text = f"{paragraph_text}\n\nThis information is from DOI: {doi}"
#             documents.append(combined_text)
#             metadata.append({"doi": doi, "source": "paragraph"})

# print(f"Total documents processed: {len(documents)}")

In [8]:
# for doc in documents:
#     print(doc)
#     print("-"*40)  # Optional: adds a visual separator between different papers


In [4]:
# ---------- Step 2: Prepare Embeddings for the Existing FAISS Index ----------
# Use the same model configuration that was used when the index was created.
# embeddings = HuggingFaceEmbeddings(model_name="allenai/scibert_scivocab_uncased")
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

/tmp/ipykernel_15670/2006348050.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")


In [5]:
# ---------- Step 2: Load the Existing Vector Database from Disk ----------
INDEX_DIRECTORY = Path("faiss_index")
INDEX_NAME = "index"
if not INDEX_DIRECTORY.exists():
    raise FileNotFoundError(f"Vector index directory '{INDEX_DIRECTORY}' not found.")

# Use allow_dangerous_deserialization=True to load indices saved with pickle-based metadata.
vector_db = FAISS.load_local(
    INDEX_DIRECTORY,
    embeddings,
    index_name=INDEX_NAME,
    allow_dangerous_deserialization=True
)
print(f"Loaded vector database from {INDEX_DIRECTORY} with {vector_db.index.ntotal} vectors")

Loaded vector database from faiss_index with 1474439 vectors


In [11]:
# # Print DOIs for up to the first 50 processed documents
# if not metadata:
#     print("No metadata available. Did you load the records?")
# else:
#     for entry in metadata[:50]:
#         print(entry.get("doi", "Unknown DOI"))

In [14]:
# ---------- Step 3: Use the Vector DB in a RAG Pipeline ----------
# Define your query.
# query = "How is hierarchical ZSM-5 synthesized?"
# query = "What is the best way to synthesize ZSM-5 that is hierarchical?"
# query = "How is Silicalite-1 synthesized?"
query = "ABW framework synthesis"


In [15]:
# Retrieve top k relevant documents (papers) from the vector database.
retrieved_docs = vector_db.similarity_search(query, k=2)
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

In [16]:
# print(embeddings.embed_query("Silicalite-1 synthesis"))
# print(embeddings.embed_documents(["test text 1", "test text 2"]))


In [17]:
context_text

"Recently, Draznieks, and Ferey et al. developed a method for computational design of inorganic framework by automated assembly of SBUs. This will provide an excellent approach to design new open-frameworks based on various SBU's of AlPOs.\n\nThis information is from DOI: 10.1021/ar0201557\n\nThe Li-A(BW) structure type (IZA code: ABW) is particularly accommodating with respect to tetrahedral-atom framework substitutions. The IZA Atlas shows that ABW isotypes containing framework-atom combinations of AlSiO, GaSiO, AlTiO, MgPO, CoPO, ZnPO, ZnAsO, BePO, and BeAsO have been prepared and characterised. CoPO-ABW phases have recently been studied in detail. These ABW-type phases fall into two classes, according to their extra-framework contents: some ABW phases contain lithium cations and extra-framework water molecules (LiABO4*H2O types), whereas others are anhydrous and contain a large univalent cation as the extra-framework species (MABO4 types; M=Rb, Cs, Tl, NH4).\n\nThis information is 

In [1]:
# RAG prompt template
RAG_PROMPT = """
Answer the question based only on the following context:
{context}
Question: {question}
Provide a detailed answer.
Provide which DOI the answer is retrieved from.
"""

# Judge prompt template for evaluation
JUDGE_PROMPT = """
You are an expert judge evaluating the quality of an answer generated by an AI system.
You will be provided with:
1. The original question
2. The context provided to the AI
3. The AI's answer

Please evaluate the answer on the following criteria:
1. Relevance (0-10): How well does the answer address the question?
2. Accuracy (0-10): How accurate is the answer based on the provided context?
3. Completeness (0-10): How complete is the answer?
4. Citation (0-10): Does it properly cite the DOI?

Question: {question}
Context: {context}
Answer: {answer}

Return ONLY a valid JSON object with no additional text or prefixes:
{{
    "relevance": {{"score": 8, "explanation": "Brief explanation here"}}, 
    "accuracy": {{"score": 9, "explanation": "Brief explanation here"}},
    "completeness": {{"score": 7, "explanation": "Brief explanation here"}},
    "citation": {{"score": 10, "explanation": "Brief explanation here"}},
    "total_score": 34,
    "overall_feedback": "Overall assessment here"
}}
"""

In [17]:
rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)
judge_prompt = ChatPromptTemplate.from_template(JUDGE_PROMPT)
prompt = rag_prompt.format(context=context_text, question=query)

In [18]:
# ---------- Step 4: Generate an Answer Using an Open-Source LLM ----------
# Setup the open source LLM using Mistral (ensure you have the model or access to it)
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False, use_auth_token=hf_token)

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v5/lib/python3.9/site-packages/transformers/models/auto/tokenization_auto.py:655: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [20]:
from transformers.models.auto.configuration_auto import CONFIG_MAPPING
print("Supported model types:", list(CONFIG_MAPPING.keys()))

Supported model types: ['albert', 'align', 'altclip', 'audio-spectrogram-transformer', 'autoformer', 'bark', 'bart', 'beit', 'bert', 'bert-generation', 'big_bird', 'bigbird_pegasus', 'biogpt', 'bit', 'blenderbot', 'blenderbot-small', 'blip', 'blip-2', 'bloom', 'bridgetower', 'bros', 'camembert', 'canine', 'chinese_clip', 'clap', 'clip', 'clipseg', 'code_llama', 'codegen', 'conditional_detr', 'convbert', 'convnext', 'convnextv2', 'cpmant', 'ctrl', 'cvt', 'data2vec-audio', 'data2vec-text', 'data2vec-vision', 'deberta', 'deberta-v2', 'decision_transformer', 'deformable_detr', 'deit', 'deta', 'detr', 'dinat', 'dinov2', 'distilbert', 'donut-swin', 'dpr', 'dpt', 'efficientformer', 'efficientnet', 'electra', 'encodec', 'encoder-decoder', 'ernie', 'ernie_m', 'esm', 'falcon', 'flaubert', 'flava', 'fnet', 'focalnet', 'fsmt', 'funnel', 'git', 'glpn', 'gpt-sw3', 'gpt2', 'gpt_bigcode', 'gpt_neo', 'gpt_neox', 'gpt_neox_japanese', 'gptj', 'gptsan-japanese', 'graphormer', 'groupvit', 'hubert', 'ibert'

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True, use_auth_token=hf_token, torch_dtype=torch.float16)

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v5/lib/python3.9/site-packages/transformers/models/auto/auto_factory.py:472: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/synthesisproject/miniforge3/envs/mrigi_tor190_v5/lib/python3.9/site-packages/transformers/utils/hub.py:374: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


In [22]:
hf_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=10000)

In [23]:
# Wrap the Hugging Face pipeline with LangChain's HuggingFacePipeline interface.
rag_llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Initialize separate judge pipeline (using same model but shorter max length for efficiency)
judge_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=2000)
judge_llm = HuggingFacePipeline(pipeline=judge_pipeline)

/tmp/ipykernel_3963756/3870768529.py:2: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  rag_llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [ ]:
# Generate the answer based on the prompt that includes retrieved context.
response_text = rag_llm(prompt)
print("Response:")
print(response_text)

/tmp/ipykernel_3963756/4206892105.py:2: LangChainDeprecationWarning: The method `BaseLLM.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response_text = rag_llm(prompt)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


# LLM Judge Evaluation System

In [ ]:
def evaluate_rag_response(query, k=2):
    # Get relevant documents
    retrieved_docs = vector_db.similarity_search(query, k=k)
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Generate RAG response
    rag_chain = LLMChain(llm=rag_llm, prompt=rag_prompt)
    response = rag_chain.run(context=context_text, question=query)
    
    # Generate evaluation
    judge_chain = LLMChain(llm=judge_llm, prompt=judge_prompt)
    evaluation = judge_chain.run(
        question=query,
        context=context_text,
        answer=response
    )
    
    # Parse the evaluation with improved JSON extraction
    try:
        # Try to extract JSON from the response
        import re
        
        # Look for JSON object starting with { and ending with }
        json_match = re.search(r'\{.*\}', evaluation, re.DOTALL)
        if json_match:
            json_str = json_match.group(0)
            evaluation_dict = json.loads(json_str)
        else:
            # If no JSON found, try parsing the whole string
            evaluation_dict = json.loads(evaluation)
            
    except (json.JSONDecodeError, AttributeError) as e:
        evaluation_dict = {
            "error": "Failed to parse evaluation",
            "raw_evaluation": evaluation,
            "parse_error": str(e)
        }
    
    return {
        "query": query,
        "context": context_text,
        "response": response,
        "evaluation": evaluation_dict,
        "timestamp": datetime.now().isoformat()
    }

In [ ]:
# List of test queries - using just the MFI alkali query for now
test_queries = [
    "How can you study theeffect of nature of silica source on the purity of template-free ZSM-5?"
]

# Run evaluation for each query
results = []
for query in tqdm(test_queries, desc="Evaluating queries"):
    result = evaluate_rag_response(query)
    results.append(result)

# Save results to JSON file
output_filename = f"rag_evaluation_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_filename, 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to {output_filename}")

Evaluating queries:   0%|          | 0/1 [00:00<?, ?it/s]/tmp/ipykernel_3715412/4288449495.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  rag_chain = LLMChain(llm=rag_llm, prompt=rag_prompt)
/tmp/ipykernel_3715412/4288449495.py:8: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = rag_chain.run(context=context_text, question=query)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/home/synthesisproject/miniforge3/envs/mrigi_tor190_v5/lib/python3.9/site-packages/transformers/generation/utils.py:1268: UserWarning: Input length of input_ids is 2060, but `max_length` is set to 2000. This can lead to unexpected behavior. You should consider increasing `max_new_tokens`.


Results saved to rag_evaluation_results_20251024_161036.json


In [ ]:
# Calculate average scores
def analyze_results(results):
    total_scores = {
        'relevance': 0,
        'accuracy': 0,
        'completeness': 0,
        'citation': 0,
        'total_score': 0
    }
    valid_results = 0
    
    for result in results:
        eval_dict = result['evaluation']
        if 'error' not in eval_dict:
            valid_results += 1
            total_scores['relevance'] += eval_dict['relevance']['score']
            total_scores['accuracy'] += eval_dict['accuracy']['score']
            total_scores['completeness'] += eval_dict['completeness']['score']
            total_scores['citation'] += eval_dict['citation']['score']
            total_scores['total_score'] += eval_dict['total_score']
    
    if valid_results > 0:
        avg_scores = {k: v/valid_results for k, v in total_scores.items()}
        print("\nAverage Scores:")
        for metric, score in avg_scores.items():
            print(f"{metric}: {score:.2f}")
    else:
        print("No valid evaluations found")

analyze_results(results)


Average Scores:
relevance: 8.00
accuracy: 9.00
completeness: 7.00
citation: 10.00
total_score: 34.00
